# Pulling Data from endpoint section

## Initial endpoint exploration

Initial setup, exploring the first page and the JSON keys and how it is structured

In [44]:
import requests
import json
import pandas as pd

In [45]:
## Initial exploration of data from importing endpoint and see which status code is returned
endpoint = 'https://europe-west3-recap-dev-347108.cloudfunctions.net/analytics-challenge-api/invoices?page=1'
response = requests.get(endpoint)
print(response.status_code)

200


In [46]:
## Retrieve sample json data from endpoint
data_import = response.json()

## Check the keys in json data
data_import.keys()

dict_keys(['data', 'page', 'page_size', 'total_items', 'total_pages'])

In [47]:
## Check data per key

for i in data_import.keys():
    print(data_import[i])

[{'company_id': 'ed811a71-439e-4c49-bfb8-8fb407c39428', 'company_name': 'DataLake Ventures', 'counter_party': 'Wholesale Dynamics GmbH', 'created_at': '2025-01-01 12:00:00', 'currency': 'EUR', 'invoice_date': '2025-01-01', 'invoice_due_date': '2025-02-15', 'invoice_id': '895a6833-d83d-4a5a-a2c3-de35b341d00a', 'invoice_number': 'INV-2025-0023', 'totalAmount': 648.15}, {'company_id': '2219af27-6da7-41a3-ac79-abce340fe56d', 'company_name': 'CloudStream Insights', 'counter_party': 'Global Services Ltd', 'created_at': '2025-01-01 08:00:00', 'currency': 'EUR', 'invoice_date': '2025-01-01', 'invoice_due_date': '2025-02-15', 'invoice_id': '4140125e-1f9b-41fb-8af2-df7996ea1e48', 'invoice_number': 'INV-2025-0239', 'totalAmount': 2141.18}, {'company_id': 'da36911c-7145-4c19-8e82-35c39f3a2e9c', 'company_name': 'AlgoRhythm Solutions', 'counter_party': 'Cloud Systems', 'created_at': '2025-01-01 10:00:00', 'currency': 'EUR', 'invoice_date': '2025-01-01', 'invoice_due_date': '2025-03-02', 'invoice_id'

In [48]:
## The key data is the total number of pages in the dataset, so we extract the value in this section:
## We set that value to "pages" and user it later.

pages = data_import['total_pages']
print (pages)

40


In [ ]:
## Visualizing the main data section in the import.

data_import = pd.json_normalize(data_import['data']) 
data_import.head()

,company_id,company_name,counter_party,created_at,currency,invoice_date,invoice_due_date,invoice_id,invoice_number,totalAmount
0,ed811a71-439e-4c49-bfb8-8fb407c39428,DataLake Ventures,Wholesale Dynamics GmbH,2025-01-01 12:00:00,EUR,2025-01-01,2025-02-15,895a6833-d83d-4a5a-a2c3-de35b341d00a,INV-2025-0023,648.15
1,2219af27-6da7-41a3-ac79-abce340fe56d,CloudStream Insights,Global Services Ltd,2025-01-01 08:00:00,EUR,2025-01-01,2025-02-15,4140125e-1f9b-41fb-8af2-df7996ea1e48,INV-2025-0239,2141.18
2,da36911c-7145-4c19-8e82-35c39f3a2e9c,AlgoRhythm Solutions,Cloud Systems,2025-01-01 10:00:00,EUR,2025-01-01,2025-03-02,abed77a5-4581-42f9-adb1-9ca99837e78f,INV-2025-0427,3474.10
3,cc007a93-8d8e-444c-bb8c-ae3db9871347,ByteForge Analytics,Enterprise Solutions AG,2025-01-01 02:00:00,GBP,2025-01-01,2025-03-02,e0e417e9-c2f2-459c-a296-2bc9ca406638,INV-2025-0498,3877.07
4,d66bfb15-e693-4219-9cb0-c5ecbb61a994,ETL Enterprises,Nordic Supply Chain,2025-01-01 05:00:00,EUR,2025-01-01,2025-01-31,095795ca-2eca-499b-805d-82d79bca6cdc,INV-2025-0691,1452.09


## Importing all endpoint data

1) We take the maximum amount of pages that exists in the keys, and loop the endpoint until the maximum page number
2) We import the data and append to a dataframe
3) There are also an extra check embeded, summing all the page_sizes in the loop to verify if its equal to ['total_items']

In [53]:
# Setting initial conditions to run loop
total_entries = 0
page_num = 1
df = pd.DataFrame()

while page_num <= pages:
    endpoint = f'https://europe-west3-recap-dev-347108.cloudfunctions.net/analytics-challenge-api/invoices?page={page_num}'
    response = requests.get(endpoint)
    data_import = response.json()
    df_temp = pd.json_normalize(data_import['data'])
    df = pd.concat([df, df_temp], ignore_index=True)
    total_entries = total_entries + data_import['page_size']
    page_num += 1

### Running Data Checks on DF

Checks to see if data export number of entries matches data

In [54]:
total_entries == data_import['total_items']

True

In [ ]:
## Checking df info after running import
# Take a look into data type outputs and if there are any null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   company_id        1000 non-null   object 
 1   company_name      1000 non-null   object 
 2   counter_party     1000 non-null   object 
 3   created_at        1000 non-null   object 
 4   currency          1000 non-null   object 
 5   invoice_date      1000 non-null   object 
 6   invoice_due_date  1000 non-null   object 
 7   invoice_id        1000 non-null   object 
 8   invoice_number    1000 non-null   object 
 9   totalAmount       1000 non-null   float64
dtypes: float64(1), object(9)
memory usage: 78.3+ KB


In [56]:
missing_values = df.isnull().sum()
print("Missing Values:")
print(missing_values)

Missing Values:
company_id          0
company_name        0
counter_party       0
created_at          0
currency            0
invoice_date        0
invoice_due_date    0
invoice_id          0
invoice_number      0
totalAmount         0
dtype: int64


### Changing data types of columns

Changing some columns to datetime values

In [57]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['invoice_due_date'] = pd.to_datetime(df['invoice_due_date'])

In [ ]:
## Checking info after data type change
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   company_id        1000 non-null   object        
 1   company_name      1000 non-null   object        
 2   counter_party     1000 non-null   object        
 3   created_at        1000 non-null   datetime64[ns]
 4   currency          1000 non-null   object        
 5   invoice_date      1000 non-null   datetime64[ns]
 6   invoice_due_date  1000 non-null   datetime64[ns]
 7   invoice_id        1000 non-null   object        
 8   invoice_number    1000 non-null   object        
 9   totalAmount       1000 non-null   float64       
dtypes: datetime64[ns](3), float64(1), object(6)
memory usage: 78.3+ KB


In [58]:
df.head()

,company_id,company_name,counter_party,created_at,currency,invoice_date,invoice_due_date,invoice_id,invoice_number,totalAmount
0,ed811a71-439e-4c49-bfb8-8fb407c39428,DataLake Ventures,Wholesale Dynamics GmbH,2025-01-01 12:00:00,EUR,2025-01-01,2025-02-15,895a6833-d83d-4a5a-a2c3-de35b341d00a,INV-2025-0023,648.15
1,2219af27-6da7-41a3-ac79-abce340fe56d,CloudStream Insights,Global Services Ltd,2025-01-01 08:00:00,EUR,2025-01-01,2025-02-15,4140125e-1f9b-41fb-8af2-df7996ea1e48,INV-2025-0239,2141.18
2,da36911c-7145-4c19-8e82-35c39f3a2e9c,AlgoRhythm Solutions,Cloud Systems,2025-01-01 10:00:00,EUR,2025-01-01,2025-03-02,abed77a5-4581-42f9-adb1-9ca99837e78f,INV-2025-0427,3474.10
3,cc007a93-8d8e-444c-bb8c-ae3db9871347,ByteForge Analytics,Enterprise Solutions AG,2025-01-01 02:00:00,GBP,2025-01-01,2025-03-02,e0e417e9-c2f2-459c-a296-2bc9ca406638,INV-2025-0498,3877.07
4,d66bfb15-e693-4219-9cb0-c5ecbb61a994,ETL Enterprises,Nordic Supply Chain,2025-01-01 05:00:00,EUR,2025-01-01,2025-01-31,095795ca-2eca-499b-805d-82d79bca6cdc,INV-2025-0691,1452.09


## Exporting Data

In [25]:
## Exporting data to a csv file

df.to_csv('endpoint_export.csv', index=False)